## Temporal split

Transactions are partitioned chronologically rather than at random, so the model is always trained on earlier activity and evaluated on later activity. This mirrors how a detection system works in practice, where only past transactions are available at the moment of scoring, and it prevents information from the evaluation period leaking into training.

The split is defined on the transaction timestamp:

- Training: the earliest 70% of transactions by time
- Validation: the following 15%
- Test: the most recent 15%

Graph features introduced later in the pipeline are derived solely from transactions that occur within the training window. Restricting the network to this period ensures that no connectivity information from the validation or test periods is encoded into a training example.

One characteristic of this dataset is worth noting. In the final days of the series, legitimate transaction volume falls sharply while the share of illicit activity rises. Because the split is chronological, these days fall in the test period, so the test set carries a markedly higher illicit rate than the training set. The chronological split is retained as specified, and absolute test-set performance is therefore read as an upper bound rather than a direct estimate of real-world detection capability.

In [2]:
# Setup Code cell

import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Fix every random source to the same seed so the results are reproducible across runs.
Seed = 42
os.environ["PYTHONHASHSEED"] = str(Seed)
random.seed(Seed)
np.random.seed(Seed)

# Define paths relative to the repository root.
# The notebook runs from the notebooks/ folder, so its parent is the project root.
ROOT = Path("..").resolve()
DATA_RAW = ROOT / "data/raw"
DATA_PROC = ROOT / "data/processed"
OUT_DIR = ROOT / "outputs"

# Create the output folders if they are not already present.
for p in [DATA_PROC, OUT_DIR / "models", OUT_DIR / "tables",
          OUT_DIR / "figures", OUT_DIR / "shap"]:
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Confirm the paths resolve correctly and the audited dataset is in place.
print("Project root:", ROOT)
print("Audited dataset found:", (DATA_PROC / "01_audit.parquet").exists())

Project root: C:\Users\chara\OneDrive\Documents\graph_AML_pipeline
Audited dataset found: True


In [4]:
# Load the audited transactions and order them chronologically.
df = pd.read_parquet(DATA_PROC / "01_audit.parquet").sort_values("Timestamp").reset_index(drop=True)

# Locate the timestamps at the 70th and 85th percentiles of the time range.
# These mark the boundaries between the training, validation, and test periods.
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)

# Label each transaction by the time window it falls into.
def assign_split(ts):
    if ts <= t70:
        return "train"
    if ts <= t85:
        return "val"
    return "test"

df["split"] = df["Timestamp"].apply(assign_split)

# Check the windows are strictly ordered in time with no overlap: the last training
# transaction comes no later than the first validation one, and likewise for validation and test.
assert df[df["split"] == "train"]["Timestamp"].max() <= df[df["split"] == "val"]["Timestamp"].min()
assert df[df["split"] == "val"]["Timestamp"].max() <= df[df["split"] == "test"]["Timestamp"].min()

# Summarise each split: row count, illicit cases, time span, and illicit rate.
summary = df.groupby("split").agg(
    n=("Is Laundering", "count"),
    illicit=("Is Laundering", "sum"),
    t_min=("Timestamp", "min"),
    t_max=("Timestamp", "max"),
).assign(illicit_rate=lambda d: d["illicit"] / d["n"])
print(summary)
summary.to_csv(OUT_DIR / "tables" / "03_split_summary.csv")

# Save the split labels so later notebooks reuse exactly the same partition.
df[["Timestamp", "split"]].to_parquet(DATA_PROC / "03_split_index.parquet")
print("\nSplit index saved to data/processed/03_split_index.parquet")

             n  illicit               t_min               t_max  illicit_rate
split                                                                        
test    761639     1561 2022-09-09 03:17:00 2022-09-18 16:18:00      0.002050
train  3554957     2856 2022-09-01 00:00:00 2022-09-07 14:55:00      0.000803
val     761749      760 2022-09-07 14:56:00 2022-09-09 03:16:00      0.000998

Split index saved to data/processed/03_split_index.parquet
